In [1]:
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [3]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [8]:
aashique = Website("https://www.emaar.com")
aashique.links

['https://properties.emaar.com/en/privacy-policy/',
 'https://properties.emaar.com/en/cookie-policy/',
 'javascript:void(0);',
 'https://properties.emaar.com/en/cookie-policy/',
 'https://properties.emaar.com/en/privacy-policy/',
 '#mobile_nav',
 'https://properties.emaar.com/en',
 'https://properties.emaar.com/en/about-emaar/',
 'https://properties.emaar.com/en/latest-launches/',
 'https://properties.emaar.com/en/our-communities/',
 'https://properties.emaar.com/en/emaar-sustainability/',
 '#',
 'https://properties.emaar.com/ar/',
 'https://properties.emaar.com/ru/',
 'https://www.dubaiemaar.cn/',
 'javascript:void(0);',
 'https://wa.link/u9ik8p',
 '#',
 'https://properties.emaar.com/our-communities/expo-living/',
 'https://properties.emaar.com/our-communities/expo-living/',
 'https://properties.emaar.com/our-communities/the-heights-country-club-wellness/',
 'https://properties.emaar.com/our-communities/the-heights-country-club-wellness/',
 'https://properties.emaar.com/our-communitie

In [9]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
        {"type": "latest projects": "url": "https://another.full.url/projects"}
    ]
}
"""

In [6]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [10]:
print(get_links_user_prompt(aashique))

Here is the list of links on the website of https://www.emaar.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://properties.emaar.com/en/privacy-policy/
https://properties.emaar.com/en/cookie-policy/
javascript:void(0);
https://properties.emaar.com/en/cookie-policy/
https://properties.emaar.com/en/privacy-policy/
#mobile_nav
https://properties.emaar.com/en
https://properties.emaar.com/en/about-emaar/
https://properties.emaar.com/en/latest-launches/
https://properties.emaar.com/en/our-communities/
https://properties.emaar.com/en/emaar-sustainability/
#
https://properties.emaar.com/ar/
https://properties.emaar.com/ru/
https://www.dubaiemaar.cn/
javascript:void(0);
https://wa.link/u9ik8p
#
https://properties.emaar.com/our-communities/expo-living/
https://properties.emaar.com/our-communities/expo-living

In [11]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [12]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [14]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
#get_brochure_user_prompt("TCS", "https://www.tcs.com")

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [29]:
#create_brochure("Mahindra", "https://www.mahindra.com")

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("Mahindra", "https://www.mahindra.com")

Found links: {'links': [{'type': 'home page', 'url': 'https://www.mahindra.com/'}, {'type': 'about page', 'url': 'https://www.mahindra.com/about-us'}, {'type': 'about our story', 'url': 'https://www.mahindra.com/about-our-story'}, {'type': 'leadership page', 'url': 'https://www.mahindra.com/leadership'}, {'type': 'careers page', 'url': 'https://www.mahindra.com/career'}, {'type': 'our business page', 'url': 'https://www.mahindra.com/our-business'}, {'type': 'our brands page', 'url': 'https://www.mahindra.com/our-brands'}, {'type': 'global presence page', 'url': 'https://www.mahindra.com/our-business/global-presence'}, {'type': 'sustainability page', 'url': 'https://www.mahindra.com/sustainability'}, {'type': 'newsroom page', 'url': 'https://www.mahindra.com/newsroom'}, {'type': 'cultural outreach page', 'url': 'https://www.mahindra.com/cultural-outreach'}, {'type': 'museum page', 'url': 'https://www.mahindra.com/museum'}, {'type': 'contact us page', 'url': 'https://www.mahindra.com/con


# Mahindra Group Brochure

## Together We Rise

### About Us
The **Mahindra Group** is a global federation of companies, recognized across diverse sectors, including automotive, farm equipment, technology services, financial services, renewable energy, logistics, hospitality, and real estate. With a presence in over **100 subsidiaries** and spanning over **20 industries**, Mahindra is driven by a core philosophy: the strength of people, the planet, and trust — **People Positive, Planet Positive, Trust Positive**.

---

### Our Vision and Commitment
At Mahindra, we believe in fostering a collaborative environment where industries and communities co-create a positive world. Our commitment to sustainability is highlighted through initiatives such as our **zero-waste approach**, successfully diverting tons of waste from landfills during our festivals.

#### Key Achievements:
- **#1 in SUVs**: 23% market share.
- **#1 in Tractors**: 44.2% market share.
- **#1 in Electric 3 Wheelers**: 41.8% market share.
- **Innovative Technology**: Launch of platforms like **IRiskMan** for better risk management.

---

### Company Culture
We cultivate a vibrant **People Positive** culture that prioritizes employee welfare and community impact. Our leadership programs, such as the **Mahindra Accelerated Leadership Track** and **Women At Mahindra**, reflect our commitment to diversity and inclusion. We encourage our employees to be leaders in their fields, supported by continuous learning and growth opportunities.

---

### Customers 
Our clientele encompasses a broad spectrum, from individual consumers of luxury SUVs to corporate clients seeking innovative tech solutions. Some of our standout brands include **XUV700** in the automotive sector, **Club Mahindra** in hospitality, and **Tech Mahindra** in technology services.

---

### Careers at Mahindra
Join us as we build a brighter future! At Mahindra, we offer various pathways to develop your career:

- **Mahindra Leadership University**: Nurturing future leaders.
- **Returnship Programs (SOAR)**: Opportunities for those reentering the workforce.
- **Innovative Roles**: Positions available across all sectors including IT, finance, and manufacturing.

Explore exciting job openings [here](https://www.mahindra.com/careers) and be part of a team that's committed to making the world a better place.

---

### Global Presence
With operations across various continents, Mahindra embodies the spirit of diverse communities. Our emphasis on local touchpoints ensures we understand and cater to the needs of our global customer base.

---

Join Mahindra on our journey towards sustainable growth and innovation. Together, we rise to create a better future for all.

**Connect with Us**:
[Website](https://www.mahindra.com) | [Investor Relations](https://www.mahindra.com/investor-relations)

---

This brochure highlights the essence of Mahindra Group, emphasizing its commitment to sustainability, innovative businesses, and a thriving company culture.